# AI HW2 — detector 


Runs tha

## 0. Settings (edit only this cell)

In [ ]:
SMOKE_TEST   = True      #
BUDGET_MIN   = 10        
SEED         = 42

COMPARE_TRAIN_STEP = 6 
TRAIN_MAX_W  = 1920     
EVAL_SET_STEP = 3        

# torchvision detectors (Faster R-CNN / RetinaNet)
TV_MIN_SIZE, TV_MAX_SIZE, TV_BATCH = 720, 1280, 2
# Ultralytics detectors (YOLO11 / RT-DETR)
DET_IMGSZ = 960
BATCH_YOLO, BATCH_RTDETR = 8, 4

RUN_TWO_STAGE = True
RUN_ONE_STAGE_ANCHOR = True
RUN_ANCHOR_FREE = True
RUN_QUERY = True
RUN_TUNED = False        # False: only the baseline of each family; True: baseline + tuned variants

# which experiments each section may run (names are defined in the sections)
FRCNN_RUN = ["frcnn_base", "frcnn_rpn3000", "frcnn_posfrac50", "frcnn_lr10"]
RET_RUN   = ["ret_base", "ret_anchors", "ret_giou", "ret_iou"]          # optional extra: "ret_focal"
AF_RUN    = ["yolo_base", "yolo_box10", "yolo_1280", "yolo_p2"]
QRY_RUN   = ["rtdetr_base", "rtdetr_nq600"]

## 1. Setup

In [ ]:
import subprocess, sys, os, json, time, math, random, shutil, zipfile, traceback
from pathlib import Path
from functools import partial

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = Path("/kaggle/input").exists() and not IN_COLAB
print("Platform:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "other")

pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.213", "pycocotools", "kaggle"], check=False)
if pip.returncode:
    print("pip install failed (on Kaggle turn Internet ON in Session options).")

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import torch, torchvision
try:
    from IPython.display import display
except ImportError:
    display = print

assert torch.cuda.is_available(), "No GPU: choose a GPU runtime."
DEV = "cuda"
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| GPU:", torch.cuda.get_device_name(0))

if SMOKE_TEST:
    BUDGET_MIN, N_EVAL, ULTRA_KW = 0.5, 24, dict(epochs=1, fraction=0.01)
else:
    N_EVAL, ULTRA_KW = None, dict(epochs=200, time=BUDGET_MIN / 60)

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/AI_HW2_T4/detectors")
    WORK = Path("/content/ai_hw2_work")
else:
    OUT_DIR = Path("/kaggle/working/detectors") if IN_KAGGLE else Path("detectors_out")
    WORK = OUT_DIR / "work"
OUT_DIR = OUT_DIR / ("smoke" if SMOKE_TEST else "final")       # smoke runs never mix with real results
for p in (OUT_DIR, WORK, OUT_DIR / "preds", OUT_DIR / "runs", OUT_DIR / "figures"):
    p.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUT_DIR / "runs"
print("Outputs:", OUT_DIR)

In [ ]:
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"
REQUIRED = ["test_images.csv", "train/annotations_coco.json", "train/images", "train/labels",
            "val/annotations_coco.json", "val/images", "val/labels"]

def is_root(p):
    return p.is_dir() and all((p / item).exists() for item in REQUIRED)

def find_root():
    bases = [Path("/kaggle/input")] if IN_KAGGLE else [Path("/content")]
    cands = [Path("/content/data"), Path("/content/dataset")] + bases
    for base in bases:
        if base.exists():
            for child in base.iterdir():
                if child.is_dir():
                    cands.append(child)
                    try:
                        cands += [g for g in child.iterdir() if g.is_dir()]
                    except PermissionError:
                        pass
    return next((p.resolve() for p in cands if is_root(p)), None)

ROOT = find_root()
if ROOT is None and IN_COLAB:
    print("Dataset not found - downloading from Kaggle (needs the KAGGLE_API_TOKEN Colab secret or kaggle.json)...")
    from google.colab import userdata
    try:
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception as e:
        raise RuntimeError("Add the KAGGLE_API_TOKEN Colab secret and enable notebook access.") from e
    subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION, "-p", "/content"], check=True)
    Path("/content/data").mkdir(exist_ok=True)
    subprocess.run(["unzip", "-q", "-n", f"/content/{COMPETITION}.zip", "-d", "/content/data"], check=True)
    ROOT = find_root()
assert ROOT is not None, "Dataset not found. On Kaggle: Add Input > Competitions > this competition."
print("Dataset root:", ROOT)

## 2. Helper functions (pure Python / NumPy)

Sampling, learning-rate schedule, anchor coverage, and an error breakdown for failure analysis. These do not need a GPU.
The error breakdown follows the idea of TIDE (Bolya et al.) in a simplified form: detections above a score threshold are matched to ground truth at IoU 0.5 and every
false positive is labelled as a classification, localization, duplicate or background error; every unmatched ground-truth box is a miss (reported by object size).

In [ ]:
def sampled_rows(df, step):
    # every `step`-th frame of each clip, frames in numeric order
    d = df.assign(_n=df["stem"].map(lambda s: int(s) if str(s).isdigit() else 0)).sort_values(["clip", "_n"])
    return pd.concat([g.iloc[::step] for _, g in d.groupby("clip", sort=False)]).drop(columns="_n")

def lr_factor(frac, it, warmup, schedule):
    # frac = elapsed / budget (0..1); linear warm-up, then cosine / step / constant
    w = min(1.0, (it + 1) / max(warmup, 1))
    if schedule == "cosine":
        s = 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(frac, 1.0)))
    elif schedule == "step":
        s = 1.0 if frac < 0.7 else 0.1
    else:
        s = 1.0
    return w * s

def set_queries(cfg, nq):
    # change the number of decoder queries in an RT-DETR model config (dict loaded from yaml)
    last = cfg["head"][-1]
    assert last[2] == "RTDETRDecoder", f"unexpected last layer: {last[2]}"
    assert last[3][2] == 300, f"expected 300 queries in the config, found {last[3][2]}"
    last[3][2] = int(nq)
    return cfg

# ---- anchor design, grounded in the object sizes -----------------------------------------------------
def anchor_shapes(sizes_per_level, ratios=(0.5, 1.0, 2.0)):
    # same convention as torchvision AnchorGenerator: ratio = h / w
    wh = []
    for sizes in sizes_per_level:
        for s in sizes:
            for r in ratios:
                wh.append((s / math.sqrt(r), s * math.sqrt(r)))
    return np.array(wh, float)

def retina_sizes(base):
    return tuple((s, int(s * 2 ** (1 / 3)), int(s * 2 ** (2 / 3))) for s in base)

def anchor_coverage(gt_wh, anchors_wh, thresholds=(0.4, 0.5)):
    # shape-only IoU (centres aligned) between every ground-truth box and its best anchor
    gw, gh = gt_wh[:, 0][:, None], gt_wh[:, 1][:, None]
    aw, ah = anchors_wh[:, 0][None, :], anchors_wh[:, 1][None, :]
    inter = np.minimum(gw, aw) * np.minimum(gh, ah)
    best = (inter / (gw * gh + aw * ah - inter)).max(1)
    return {f"cover@{t}": round(float((best >= t).mean()), 3) for t in thresholds} | {"median_best_iou": round(float(np.median(best)), 3)}

#  error breakdown 
def xywh_to_xyxy(b):
    b = np.asarray(b, float).reshape(-1, 4)
    return np.stack([b[:, 0], b[:, 1], b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]], 1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / np.maximum(area(a)[:, None] + area(b)[None, :] - inter, 1e-9)

def build_gt_index(dataset):
    per = {}
    for a in dataset["annotations"]:
        per.setdefault(a["image_id"], []).append(a)
    gt, crowd = {}, {}
    for iid, anns in per.items():
        reg = [a for a in anns if not a.get("iscrowd", 0)]
        cr = [a for a in anns if a.get("iscrowd", 0)]
        gt[iid] = (np.array([a["bbox"] for a in reg], float).reshape(-1, 4), np.array([a["category_id"] for a in reg], int))
        crowd[iid] = np.array([a["bbox"] for a in cr], float).reshape(-1, 4)
    return gt, crowd

FP_TYPES = ["FP_classification", "FP_localization", "FP_duplicate", "FP_background"]

def error_breakdown(results, gt, crowd, image_ids, score_thr=0.3, iou_thr=0.5):
    by_img = {}
    for r in results:
        if r["score"] >= score_thr:
            by_img.setdefault(r["image_id"], []).append(r)
    totals = {k: 0 for k in ["TP", "FN"] + FP_TYPES}
    size_all, size_hit = np.zeros(3), np.zeros(3)
    rows, detail = [], {}
    for iid in image_ids:
        gb, gc = gt.get(iid, (np.zeros((0, 4)), np.zeros(0, int)))
        gx = xywh_to_xyxy(gb)
        cr = xywh_to_xyxy(crowd.get(iid, np.zeros((0, 4))))
        dets = sorted(by_img.get(iid, []), key=lambda r: -r["score"])
        db = xywh_to_xyxy([d["bbox"] for d in dets]); dc = np.array([d["category_id"] for d in dets], int)
        if len(cr) and len(db):                                    # detections centred in an ignored region are not counted
            cx, cy = (db[:, 0] + db[:, 2]) / 2, (db[:, 1] + db[:, 3]) / 2
            inside = ((cx[:, None] >= cr[None, :, 0]) & (cx[:, None] <= cr[None, :, 2]) &
                      (cy[:, None] >= cr[None, :, 1]) & (cy[:, None] <= cr[None, :, 3])).any(1)
            db, dc = db[~inside], dc[~inside]
        iou = iou_matrix(db, gx)
        matched = np.zeros(len(gx), bool); types = []
        for i in range(len(db)):
            same = gc == dc[i]
            cand = np.where(same & ~matched & (iou[i] >= iou_thr))[0] if len(gx) else np.array([], int)
            if len(cand):
                matched[cand[np.argmax(iou[i][cand])]] = True; types.append("TP"); continue
            if len(gx) and (iou[i][~same] >= iou_thr).any():
                t = "FP_classification"
            elif len(gx) and same.any() and iou[i][same].max() >= iou_thr:
                t = "FP_duplicate"
            elif len(gx) and same.any() and iou[i][same].max() >= 0.1:
                t = "FP_localization"
            else:
                t = "FP_background"
            types.append(t)
        types = np.array(types, dtype=object)
        gsize = np.sqrt(gb[:, 2] * gb[:, 3]) if len(gb) else np.zeros(0)
        gbin = (gsize >= 32).astype(int) + (gsize >= 96).astype(int)
        size_all += np.bincount(gbin, minlength=3); size_hit += np.bincount(gbin[matched], minlength=3)
        counts = {k: int((types == k).sum()) for k in ["TP"] + FP_TYPES}
        counts["FN"] = int((~matched).sum())
        for k, v in counts.items():
            totals[k] += v
        rows.append(dict(image_id=iid, n_gt=len(gx), recall=float(matched.mean()) if len(gx) else np.nan, **counts))
        detail[iid] = dict(det=db, det_type=types, gt=gx, matched=matched)
    size_recall = {n: round(float(h / a), 3) if a else np.nan for n, h, a in zip(["small<32", "medium32-96", "large>96"], size_hit, size_all)}
    return totals, size_recall, pd.DataFrame(rows), detail

def pick_failures(per_img, min_gt=20):
    d = per_img[per_img["n_gt"] >= min_gt]
    cases = {}
    cases["lowest recall"] = int(d.sort_values("recall").iloc[0]["image_id"])
    cases["most background FPs"] = int(d.sort_values("FP_background", ascending=False).iloc[0]["image_id"])
    cases["most class confusions"] = int(d.sort_values("FP_classification", ascending=False).iloc[0]["image_id"])
    return cases

COLORS = {"FP_classification": "magenta", "FP_localization": "orange", "FP_duplicate": "cyan", "FP_background": "red"}

def show_failures(cases, detail, path_by_id, name, max_w=900):
    fig, axes = plt.subplots(1, len(cases), figsize=(7 * len(cases), 4.6))
    axes = np.atleast_1d(axes)
    for ax, (title, iid) in zip(axes, cases.items()):
        im = Image.open(path_by_id[iid]).convert("RGB"); f = min(1.0, max_w / im.size[0])
        ax.imshow(im.resize((int(im.size[0] * f), int(im.size[1] * f)))); ax.axis("off")
        d = detail[iid]
        for b, m in zip(d["gt"] * f, d["matched"]):
            if not m:
                ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="yellow", lw=1.2))
        db = d["det"] * f
        for b, t in zip(db, d["det_type"]):
            if t != "TP":
                ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=COLORS[t], lw=1.0))
        ax.set_title(f"{title} (image {iid})", fontsize=10)
    fig.suptitle(f"{name}: yellow = missed GT; " + ", ".join(f"{c} = {k}" for k, c in COLORS.items()), fontsize=9)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / f"failures_{name}.png", dpi=130); plt.show()

## 3. Shared protocol: frames, ground truth, evaluator

Training frames come from the YOLO label files (class, cx, cy, w, h normalised); ignored regions are **not** used in training by any model here (same as the main notebook).
The evaluator uses the COCO annotation file of the val split, where ignored regions are `iscrowd=1`, so detections inside them are not penalised.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import io, contextlib

NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]    # COCO category_id 1..8
TRAIN_NAMES = NAMES.copy(); TRAIN_NAMES[7] = "motorcycle"                         # pretrained-COCO class name for id 8

def _resolve(split, fn):
    for c in (ROOT / split / fn, ROOT / fn, ROOT / split / "images" / fn):
        if c.exists():
            return str(c.resolve())
    return None

# ---- validation frames + COCO ground truth
with contextlib.redirect_stdout(io.StringIO()):
    coco_gt = COCO(str(ROOT / "val" / "annotations_coco.json"))
for a in coco_gt.dataset["annotations"]:
    a.setdefault("area", a["bbox"][2] * a["bbox"][3])
val_df = pd.DataFrame(coco_gt.dataset["images"])[["id", "file_name", "width", "height"]].rename(columns={"id": "image_id"})
val_df["path"] = [_resolve("val", f) for f in val_df["file_name"]]
val_df = val_df.dropna(subset=["path"]).reset_index(drop=True)
val_df["clip"] = val_df["path"].map(lambda p: Path(p).parent.name)
val_df["stem"] = val_df["path"].map(lambda p: Path(p).stem)

EVAL_ROWS = sampled_rows(val_df[val_df["width"] <= 1904], EVAL_SET_STEP).reset_index(drop=True)
if N_EVAL:
    EVAL_ROWS = EVAL_ROWS.iloc[::max(1, len(EVAL_ROWS) // N_EVAL)].head(N_EVAL).reset_index(drop=True)
PATH_BY_ID = dict(zip(val_df["image_id"], val_df["path"]))
GT_IDX, CROWD_IDX = build_gt_index(coco_gt.dataset)
EVAL_COUNTS = np.array([len(GT_IDX.get(i, (np.zeros((0, 4)),))[0]) for i in EVAL_ROWS["image_id"]])
print(f"Evaluation frames: {len(EVAL_ROWS)} | objects per frame: median {int(np.median(EVAL_COUNTS))}, max {int(EVAL_COUNTS.max())}")

#  training frames (sizes read from the image header, labels from the YOLO txt files)
rows = []
for p in sorted((ROOT / "train" / "images").glob("*/*.jpg")):
    rows.append(dict(path=str(p.resolve()), clip=p.parent.name, stem=p.stem))
train_all = pd.DataFrame(rows)
train_df = sampled_rows(train_all, COMPARE_TRAIN_STEP).reset_index(drop=True)
train_df[["width", "height"]] = [Image.open(p).size for p in train_df["path"]]
train_df = train_df[train_df["width"] <= TRAIN_MAX_W].reset_index(drop=True)
if SMOKE_TEST:
    train_df = train_df.iloc[::max(1, len(train_df) // 200)].reset_index(drop=True)
print(f"Train frames: {len(train_df):,} (stride {COMPARE_TRAIN_STEP}, width <= {TRAIN_MAX_W}) from {train_df['clip'].nunique()} clips")

def read_yolo_labels(split, clip, stem, W, H):
    p = ROOT / split / "labels" / clip / f"{stem}.txt"
    boxes, labels = [], []
    if p.exists():
        for line in p.read_text().splitlines():
            v = line.split()
            if len(v) != 5:
                continue
            c, cx, cy, bw, bh = int(float(v[0])), *map(float, v[1:])
            x1, y1, x2, y2 = (cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H
            if x2 - x1 >= 1 and y2 - y1 >= 1:
                boxes.append([x1, y1, x2, y2]); labels.append(c + 1)           # labels 1..8 = COCO category ids
    return np.array(boxes, np.float32).reshape(-1, 4), np.array(labels, np.int64)

TRAIN_ANN = [read_yolo_labels("train", r.clip, r.stem, r.width, r.height) for r in train_df.itertuples()]
keep = [i for i, (b, _) in enumerate(TRAIN_ANN) if len(b)]
train_df = train_df.iloc[keep].reset_index(drop=True); TRAIN_ANN = [TRAIN_ANN[i] for i in keep]
print("Train frames with objects:", len(train_df))

#  lists + data.yaml for the Ultralytics models
(WORK / "train_list.txt").write_text("\n".join(train_df["path"]))
(WORK / "val_list.txt").write_text("\n".join(EVAL_ROWS["path"].iloc[::2]))             # lighter per-epoch validation
DATA_YAML = WORK / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path": str(ROOT), "train": str(WORK / "train_list.txt"), "val": str(WORK / "val_list.txt"),
                                     "names": {i: n for i, n in enumerate(TRAIN_NAMES)}}, sort_keys=False))

In [ ]:
def coco_eval(results, image_ids, max_det=500):
    # COCO-style AP / AP50 / AP_small ... on the given image ids; ignored regions are iscrowd=1 in the ground truth
    if not results:
        return dict(AP=0.0, AP50=0.0, AP75=0.0, AP_small=0.0, AP_medium=0.0, AP_large=0.0, AR=0.0), {n: float("nan") for n in NAMES}
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(coco_gt, coco_gt.loadRes(results), "bbox")
        E.params.imgIds = [int(i) for i in image_ids]
        E.params.maxDets = [1, 10, max_det]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    summary = dict(AP=s[0], AP50=s[1], AP75=s[2], AP_small=s[3], AP_medium=s[4], AP_large=s[5], AR=s[8])
    prec = E.eval["precision"]
    per_class = {}
    for k, cid in enumerate(E.params.catIds):
        p = prec[:, :, k, 0, -1]; p = p[p > -1]
        per_class[NAMES[cid - 1]] = float(p.mean()) if p.size else float("nan")
    return {k: float(v) for k, v in summary.items()}, per_class

def predict_ultra(model, rows, imgsz, iou=0.7, conf=0.001, max_det=500, batch=4):
    out, paths, ids = [], rows["path"].tolist(), rows["image_id"].tolist()
    t_total = 0.0
    for s in range(0, len(paths), batch):
        torch.cuda.synchronize(); t = time.perf_counter()
        res = model.predict(paths[s:s + batch], imgsz=imgsz, conf=conf, iou=iou, max_det=max_det, batch=batch, device=0, verbose=False)
        torch.cuda.synchronize(); t_total += time.perf_counter() - t
        for iid, r in zip(ids[s:s + batch], res):
            if r.boxes is None or len(r.boxes) == 0:
                continue
            xyxy, sc = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
            cl = r.boxes.cls.cpu().numpy().astype(int)
            for (x1, y1, x2, y2), score, c in zip(xyxy[np.argsort(-sc)[:max_det]], np.sort(sc)[::-1][:max_det], cl[np.argsort(-sc)[:max_det]]):
                out.append(dict(image_id=int(iid), category_id=int(c) + 1, bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)], score=float(score)))
    return out, 1000 * t_total / len(paths)      # ms per image, includes pre/post-processing

#  registry: every finished run is appended and saved
REG_PATH = OUT_DIR / "registry.json"
REGISTRY = json.loads(REG_PATH.read_text()) if REG_PATH.exists() else []

def done(name):
    return any(r["name"] == name for r in REGISTRY)

def log_result(family, name, metrics, ms, steps, params, per_class=None, extra=None):
    REGISTRY[:] = [r for r in REGISTRY if r["name"] != name]
    REGISTRY.append(dict(family=family, name=name, **{k: round(v, 4) for k, v in metrics.items()}, ms_per_img=round(ms, 1),
                         budget_min=BUDGET_MIN, steps=steps, params=params, per_class=per_class or {}, extra=extra or {}))
    REG_PATH.write_text(json.dumps(REGISTRY, indent=1))

def save_preds(name, results):
    (OUT_DIR / "preds" / f"{name}.json").write_text(json.dumps([r for r in results if r["score"] >= 0.05]))

def load_preds(name):
    return json.loads((OUT_DIR / "preds" / f"{name}.json").read_text())

def show_registry(family=None):
    df = pd.DataFrame(REGISTRY)
    if df.empty:
        print("no finished runs yet"); return df
    if family:
        df = df[df["family"] == family]
    cols = ["family", "name", "AP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img", "steps", "params"]
    df = df[cols].set_index("name"); display(df); return df

def free_gpu():
    import gc; gc.collect(); torch.cuda.empty_cache()

## 4. torchvision training / evaluation utilities

Both torchvision detectors start from COCO-pretrained weights, get a new head for 8 classes (labels 1..8 = competition category ids; label 0 is unused) and train with
SGD, mixed precision, horizontal flip, linear warm-up, so every model gets the same compute.
Images are resized inside the model (`min_size`/`max_size`); predictions come back in original pixel coordinates.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image, ImageReadMode
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, retinanet_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNet_ResNet50_FPN_V2_Weights, RetinaNetClassificationHead
from torchvision.models.detection.anchor_utils import AnchorGenerator
import torchvision.models.detection.retinanet as retinanet_module
import torch.nn as nn

class TrainSet(Dataset):
    def __init__(self, df, ann):
        self.df, self.ann = df, ann
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        img = read_image(self.df.iloc[i]["path"], mode=ImageReadMode.RGB).float().div(255)
        boxes, labels = self.ann[i]
        boxes = torch.from_numpy(boxes.copy()); labels = torch.from_numpy(labels.copy())
        if random.random() < 0.5:                                     # horizontal flip
            W = img.shape[-1]; img = img.flip(-1)
            boxes = torch.stack([W - boxes[:, 2], boxes[:, 1], W - boxes[:, 0], boxes[:, 3]], 1) if len(boxes) else boxes
        return img, {"boxes": boxes, "labels": labels}

TRAIN_DS = TrainSet(train_df, TRAIN_ANN)
collate = lambda b: tuple(zip(*b))

def train_tv(model, budget_min, lr, schedule="cosine", warmup=100, wd=1e-4):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)
    scaler = torch.amp.GradScaler("cuda")
    dl = DataLoader(TRAIN_DS, batch_size=TV_BATCH, shuffle=True, num_workers=2, collate_fn=collate, drop_last=True)
    model.train(); t0 = time.time(); it = 0; hist = []; finished = False
    while not finished:
        for imgs, tgts in dl:
            frac = (time.time() - t0) / (budget_min * 60)
            if frac >= 1:
                finished = True; break
            for g in opt.param_groups:
                g["lr"] = lr * lr_factor(frac, it, warmup, schedule)
            imgs = [i.to(DEV, non_blocking=True) for i in imgs]
            tgts = [{k: v.to(DEV) for k, v in t.items()} for t in tgts]
            with torch.autocast("cuda", dtype=torch.float16):
                losses = model(imgs, tgts)
            loss = sum(losses.values())
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 10.0)
            scaler.step(opt); scaler.update(); it += 1
            if it % 25 == 0:
                hist.append((it, float(loss)))
                print(f"  it {it:5d}  loss {float(loss):.3f}  lr {opt.param_groups[0]['lr']:.5f}  {frac * 100:.0f}% of budget", end="\r")
    print()
    return dict(iters=it, loss_curve=hist)

@torch.no_grad()
def predict_tv(model, rows, batch=2, max_det=500):
    model.eval(); out, t_model = [], 0.0
    paths, ids = rows["path"].tolist(), rows["image_id"].tolist()
    for s in range(0, len(paths), batch):
        imgs = [read_image(p, mode=ImageReadMode.RGB).float().div(255).to(DEV) for p in paths[s:s + batch]]
        torch.cuda.synchronize(); t = time.perf_counter()
        with torch.autocast("cuda", dtype=torch.float16):
            preds = model(imgs)
        torch.cuda.synchronize(); t_model += time.perf_counter() - t
        for iid, p in zip(ids[s:s + batch], preds):
            b, sc, lb = p["boxes"].float().cpu().numpy(), p["scores"].float().cpu().numpy(), p["labels"].cpu().numpy()
            for j in np.argsort(-sc)[:max_det]:
                if 1 <= lb[j] <= 8:
                    x1, y1, x2, y2 = b[j]
                    out.append(dict(image_id=int(iid), category_id=int(lb[j]), bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)], score=float(sc[j])))
    return out, 1000 * t_model / len(paths)                    # ms per image, model forward only

def run_tv(family, name, build, lr=0.005, schedule="cosine", params="", post_eval=None):
    if done(name):
        print(f"[skip] {name} is already in the registry"); return
    print(f"=== {name} | {params} | budget {BUDGET_MIN} min")
    try:
        seed_all(); model = build().to(DEV)
        tr = train_tv(model, BUDGET_MIN, lr, schedule)
        res, ms = predict_tv(model, EVAL_ROWS)
        m, pc = coco_eval(res, EVAL_ROWS["image_id"])
        extra = post_eval(model) if post_eval else None
        save_preds(name, res)
        log_result(family, name, m, ms, f"{tr['iters']} it x {TV_BATCH}", params, pc, extra)
        print(f"  AP {m['AP']:.4f}  AP50 {m['AP50']:.4f}  AP_small {m['AP_small']:.4f}  {ms:.0f} ms/img  ({tr['iters']} iterations)")
    except Exception:
        traceback.print_exc(limit=3); print(f"  !! {name} failed, continuing")
    finally:
        try:
            del model
        except NameError:
            pass
        free_gpu()

## 5. Two-stage anchor-based: Faster R-CNN

**Setup.** `fasterrcnn_resnet50_fpn_v2`, COCO-pretrained, new 8-class box predictor, FPN features, resized to `TV_MAX_SIZE` on the long side, SGD (momentum 0.9), AMP, horizontal flip.

**Parameters investigated** (search range -> baseline first):

| experiment | change | range / value |
|---|---|---|
| `frcnn_base` | RPN keeps 2000 proposals in training, 1000 at test; RoI head samples 512 RoIs with 25 % positives; LR 0.005 cosine | baseline |
| `frcnn_rpn3000` | number of RPN proposals (scenes can hold hundreds of objects) | train 2000 -> 3000, test 1000 -> 2000 |
| `frcnn_posfrac50` | RoI sampling: share of positive RoIs per image (dense scenes have many positives) | 0.25 -> 0.5 |
| `frcnn_lr10` | learning rate / schedule | 0.005 -> 0.01 |

In [ ]:
def build_frcnn(**kw):
    def _b():
        m = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, min_size=TV_MIN_SIZE, max_size=TV_MAX_SIZE,
                                       box_detections_per_img=500, box_score_thresh=0.001,
                                       rpn_pre_nms_top_n_train=kw.get("pre_tr", 2000), rpn_post_nms_top_n_train=kw.get("post_tr", 2000),
                                       rpn_pre_nms_top_n_test=kw.get("pre_te", 1000), rpn_post_nms_top_n_test=kw.get("post_te", 1000),
                                       box_batch_size_per_image=kw.get("roi_batch", 512), box_positive_fraction=kw.get("pos_frac", 0.25))
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 9)
        return m
    return _b

@torch.no_grad()
def rpn_recall(model, n_images=60, ns=(100, 300, 1000), iou_thr=0.5):
    # share of ground-truth boxes covered (IoU >= 0.5) by the top-N RPN proposals
    model.eval()
    hit = {n: np.zeros(3) for n in ns}; tot = np.zeros(3)
    sel = EVAL_ROWS.iloc[::max(1, len(EVAL_ROWS) // n_images)].head(n_images)
    for r in sel.itertuples():
        gb, _ = GT_IDX.get(r.image_id, (np.zeros((0, 4)), None))
        if len(gb) == 0:
            continue
        xyxy = xywh_to_xyxy(gb).astype(np.float32)
        size = np.sqrt(gb[:, 2] * gb[:, 3]); sbin = (size >= 32).astype(int) + (size >= 96).astype(int)
        img = read_image(r.path, mode=ImageReadMode.RGB).float().div(255).to(DEV)
        il, tl = model.transform([img], [{"boxes": torch.from_numpy(xyxy).to(DEV), "labels": torch.ones(len(xyxy), dtype=torch.int64, device=DEV)}])
        props, _ = model.rpn(il, model.backbone(il.tensors))
        iou = torchvision.ops.box_iou(tl[0]["boxes"], props[0]).cpu().numpy()
        tot += np.bincount(sbin, minlength=3)
        for n in ns:
            best = iou[:, :n].max(1) if iou.shape[1] else np.zeros(len(xyxy))
            hit[n] += np.bincount(sbin[best >= iou_thr], minlength=3)
    out = {}
    for n in ns:
        out[f"recall@{n}"] = round(float(hit[n].sum() / tot.sum()), 3)
        out[f"recall@{n}_small"] = round(float(hit[n][0] / tot[0]), 3) if tot[0] else None
    return out

FRCNN = {
    "frcnn_base":      dict(build=build_frcnn(), lr=0.005, params="RPN 2000/1000, RoI 512 @ 25% pos, lr 0.005"),
    "frcnn_rpn3000":   dict(build=build_frcnn(pre_tr=3000, post_tr=3000, pre_te=3000, post_te=2000), lr=0.005, params="RPN 3000/2000, RoI 512 @ 25% pos, lr 0.005"),
    "frcnn_posfrac50": dict(build=build_frcnn(pos_frac=0.5), lr=0.005, params="RPN 2000/1000, RoI 512 @ 50% pos, lr 0.005"),
    "frcnn_lr10":      dict(build=build_frcnn(), lr=0.01, params="RPN 2000/1000, RoI 512 @ 25% pos, lr 0.01"),
}
if RUN_TWO_STAGE:
    for name in FRCNN_RUN:
        if name != "frcnn_base" and not RUN_TUNED:
            continue
        cfg = FRCNN[name]
        run_tv("two-stage", name, cfg["build"], lr=cfg["lr"], params=cfg["params"],
               post_eval=rpn_recall if name == "frcnn_base" else None)
show_registry("two-stage");

### Two-stage: what each stage contributes and where it fails

The proposal-recall numbers come from the baseline run (`extra` in the registry): they show how many ground-truth boxes the RPN can cover *before* the second stage classifies and refines anything.
The error breakdown below uses detections with score >= 0.3 at IoU 0.5.

In [ ]:
base = next((r for r in REGISTRY if r["name"] == "frcnn_base"), None)
if base:
    print("RPN proposal recall (IoU >= 0.5) of the baseline:", base["extra"])
    ft, size_rec, per_img, detail = error_breakdown(load_preds("frcnn_base"), GT_IDX, CROWD_IDX, EVAL_ROWS["image_id"])
    display(pd.Series(ft, name="count").to_frame().T)
    print("recall at score >= 0.3 by object size:", size_rec)
    cases = pick_failures(per_img)
    print("failure cases:", cases)
    display(per_img.set_index("image_id").loc[list(cases.values())])
    show_failures(cases, detail, PATH_BY_ID, "frcnn_base")

### Your observations — two-stage

- What does the RPN recall say about small objects and about the number of proposals? What would the second stage still have to fix?
- Compare the tuned runs with the baseline: which parameter mattered, and is the difference larger than run-to-run noise (short budget, one seed)?
- For each of the three failure images: what is wrong (missed, background, confusion), and what plausible cause fits (tiny objects, occlusion, crowded scene, similar classes such as van/car/truck, ignored regions)?

## 6. One-stage anchor-based: RetinaNet

**Setup.** `retinanet_resnet50_fpn_v2`, COCO-pretrained, new 8-class classification head (the regression head keeps its weights), same training protocol as Faster R-CNN.

**Anchors grounded in the data.** Anchor sizes should match the object sizes *at network resolution* (objects are scaled by `TV_MAX_SIZE / long side`). The next cell measures how many training
boxes have an anchor of IoU >= 0.5 (the positive-assignment threshold) for different configurations. Default RetinaNet uses sizes 32-512 on pyramid levels P3-P7 (3 scales x 3 aspect ratios).

In [ ]:
sc = np.array([min(TV_MIN_SIZE / min(w, h), TV_MAX_SIZE / max(w, h)) for w, h in zip(train_df["width"], train_df["height"])])
gt_wh = np.concatenate([np.stack([(b[:, 2] - b[:, 0]) * s, (b[:, 3] - b[:, 1]) * s], 1) for (b, _), s in zip(TRAIN_ANN, sc)])
print("object size at network resolution (px): p05 %.1f | p25 %.1f | median %.1f | p75 %.1f | p95 %.1f" %
      tuple(np.percentile(np.sqrt(gt_wh[:, 0] * gt_wh[:, 1]), [5, 25, 50, 75, 95])))

ANCHOR_CONFIGS = {"default 32-512": (32, 64, 128, 256, 512), "16-256": (16, 32, 64, 128, 256), "12-192": (12, 24, 48, 96, 192), "8-128": (8, 16, 32, 64, 128)}
cov = {k: anchor_coverage(gt_wh, anchor_shapes(retina_sizes(v)), thresholds=(0.4, 0.5)) for k, v in ANCHOR_CONFIGS.items()}
display(pd.DataFrame(cov).T)

**Experiments** (incremental, each adds one change to the previous one):

| experiment | change |
|---|---|
| `ret_base` | default anchors 32-512, L1 box loss, focal alpha 0.25 / gamma 2, IoU thresholds 0.5 / 0.4 |
| `ret_anchors` | anchor sizes 16-256 (chosen from the coverage table above) |
| `ret_giou` | `ret_anchors` + GIoU box-regression loss instead of L1 |
| `ret_iou` | `ret_giou` + lower assignment thresholds 0.4 / 0.3 (more positives for small objects) |
| `ret_focal` (optional) | `ret_giou` + focal-loss parameters (range: gamma 1.0-2.5, alpha 0.25-0.5) |

Edit `ANCHOR_BASE_TUNED` if the coverage table suggests a better configuration.

In [ ]:
ANCHOR_BASE_TUNED = (16, 32, 64, 128, 256)
FOCAL = dict(alpha=0.25, gamma=2.0)

# torchvision's RetinaNet calls sigmoid_focal_loss with its defaults; route the call through a wrapper to tune alpha / gamma
if hasattr(retinanet_module, "sigmoid_focal_loss") and not getattr(retinanet_module, "_focal_patched", False):
    _orig_focal = retinanet_module.sigmoid_focal_loss
    def _focal(inputs, targets, alpha=0.25, gamma=2.0, reduction="none"):
        return _orig_focal(inputs, targets, alpha=FOCAL["alpha"], gamma=FOCAL["gamma"], reduction=reduction)
    retinanet_module.sigmoid_focal_loss = _focal
    retinanet_module._focal_patched = True
else:
    print("focal-loss patch not applied (name not found or already patched): ret_focal would equal ret_giou")

def build_retina(anchor_base=(32, 64, 128, 256, 512), reg_loss="l1", fg=0.5, bg=0.4, focal=(0.25, 2.0)):
    def _b():
        FOCAL.update(alpha=focal[0], gamma=focal[1])
        m = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1, min_size=TV_MIN_SIZE, max_size=TV_MAX_SIZE,
                                      detections_per_img=500, score_thresh=0.001, topk_candidates=1000, fg_iou_thresh=fg, bg_iou_thresh=bg)
        sizes = retina_sizes(anchor_base)
        m.anchor_generator = AnchorGenerator(sizes, ((0.5, 1.0, 2.0),) * len(sizes))
        A = m.anchor_generator.num_anchors_per_location()[0]
        m.head.classification_head = RetinaNetClassificationHead(256, A, 9, norm_layer=partial(nn.GroupNorm, 32))
        assert hasattr(m.head.regression_head, "_loss_type"), "this torchvision has no configurable box loss"
        m.head.regression_head._loss_type = reg_loss
        return m
    return _b

RET = {
    "ret_base":    dict(build=build_retina(), params="anchors 32-512, L1, IoU 0.5/0.4, focal 0.25/2"),
    "ret_anchors": dict(build=build_retina(ANCHOR_BASE_TUNED), params=f"anchors {ANCHOR_BASE_TUNED[0]}-{ANCHOR_BASE_TUNED[-1]}, L1, IoU 0.5/0.4"),
    "ret_giou":    dict(build=build_retina(ANCHOR_BASE_TUNED, "giou"), params="tuned anchors, GIoU loss, IoU 0.5/0.4"),
    "ret_iou":     dict(build=build_retina(ANCHOR_BASE_TUNED, "giou", 0.4, 0.3), params="tuned anchors, GIoU, IoU 0.4/0.3"),
    "ret_focal":   dict(build=build_retina(ANCHOR_BASE_TUNED, "giou", focal=(0.5, 1.5)), params="tuned anchors, GIoU, focal 0.5/1.5"),
}
if RUN_ONE_STAGE_ANCHOR:
    for name in RET_RUN:
        if name != "ret_base" and not RUN_TUNED:
            continue
        run_tv("one-stage anchor-based", name, RET[name]["build"], lr=0.005, params=RET[name]["params"])
show_registry("one-stage anchor-based");

### one-stage anchor-based

- Which anchor configuration covers the data best, and did better coverage translate into better AP_small?
- Removing the proposal stage means every anchor location is a candidate, so background anchors vastly outnumber objects. How does focal loss address this, and what did changing its parameters or the assignment thresholds do?
- Compare initial and tuned RetinaNet with the tuned Faster R-CNN (accuracy, AP_small, ms/img).

## 7. Anchor-free: YOLO11 (tuned, results by object size)

YOLO11 has no anchors: every feature-map location is a candidate whose box is predicted as distances to the four edges (distribution focal loss). Positives are chosen per object by a task-aligned
assigner that scores each location by predicted class score and IoU with the object, so assignment adapts to size and shape (the "label assignment" part of the rubric — read the TOOD paper in the reading list).

**Parameters investigated** (all at the same wall-clock budget):

| experiment | change | range |
|---|---|---|
| `yolo_base` | YOLO11s, `DET_IMGSZ`, loss gains box 7.5 / cls 0.5 / dfl 1.5 | baseline |
| `yolo_box10` | box-regression loss weight | 7.5 -> 10 |
| `yolo_1280` | input resolution | `DET_IMGSZ` -> 1280 (batch halves) |
| `yolo_p2` | extra stride-4 (P2) high-resolution detection head | needs `yolo11s-p2.yaml` in your Ultralytics version |

In [ ]:
from ultralytics import YOLO, RTDETR

def run_ultra(family, name, make_model, cls, imgsz, batch, lr0, extra=None, params=""):
    if done(name):
        print(f"[skip] {name} is already in the registry"); return
    print(f"=== {name} | {params} | budget {BUDGET_MIN} min")
    try:
        seed_all(); model = make_model()
        model.train(data=str(DATA_YAML), imgsz=imgsz, batch=batch, optimizer="AdamW", lr0=lr0, lrf=0.1, cos_lr=True, warmup_epochs=0.5,
                    close_mosaic=1, mosaic=1.0, degrees=0.0, scale=0.3, patience=0, plots=False, workers=2, device=0, amp=True,
                    max_det=500, seed=SEED, deterministic=False, project=str(RUNS_DIR), name=name, exist_ok=True, **ULTRA_KW, **(extra or {}))
        del model; free_gpu()
        det = cls(str(RUNS_DIR / name / "weights" / "last.pt"))
        res, ms = predict_ultra(det, EVAL_ROWS, imgsz, batch=4)
        m, pc = coco_eval(res, EVAL_ROWS["image_id"])
        n_ep = len(pd.read_csv(RUNS_DIR / name / "results.csv"))
        save_preds(name, res)
        log_result(family, name, m, ms, f"{n_ep} epochs", params, pc)
        print(f"  AP {m['AP']:.4f}  AP50 {m['AP50']:.4f}  AP_small {m['AP_small']:.4f}  {ms:.0f} ms/img  ({n_ep} epochs)")
        del det
    except Exception:
        traceback.print_exc(limit=3); print(f"  !! {name} failed, continuing")
    finally:
        free_gpu()

AF = {
    "yolo_base":  dict(make=lambda: YOLO("yolo11s.pt"), imgsz=DET_IMGSZ, batch=BATCH_YOLO, extra={}, params=f"imgsz {DET_IMGSZ}, box 7.5, cls 0.5"),
    "yolo_box10": dict(make=lambda: YOLO("yolo11s.pt"), imgsz=DET_IMGSZ, batch=BATCH_YOLO, extra=dict(box=10.0), params=f"imgsz {DET_IMGSZ}, box 10"),
    "yolo_1280":  dict(make=lambda: YOLO("yolo11s.pt"), imgsz=1280, batch=BATCH_YOLO // 2, extra={}, params="imgsz 1280, box 7.5"),
    "yolo_p2":    dict(make=lambda: YOLO("yolo11s-p2.yaml").load("yolo11s.pt"), imgsz=DET_IMGSZ, batch=BATCH_YOLO, extra={}, params=f"imgsz {DET_IMGSZ}, extra P2 head"),
}
if RUN_ANCHOR_FREE:
    for name in AF_RUN:
        if name != "yolo_base" and not RUN_TUNED:
            continue
        c = AF[name]
        run_ultra("anchor-free", name, c["make"], YOLO, c["imgsz"], c["batch"], 0.0005, c["extra"], c["params"])
show_registry("anchor-free");

In [ ]:
# results broken down by object size and by class, for every finished model so far (the rubric asks for this comparison)
tab = pd.DataFrame(REGISTRY)
if not tab.empty:
    by_size = tab.set_index("name")[["family", "AP", "AP_small", "AP_medium", "AP_large", "ms_per_img"]]
    display(by_size.round(4))
    display(pd.DataFrame({r["name"]: r["per_class"] for r in REGISTRY}).round(3))

### anchor-free

- How are positive locations chosen for an object without anchors, and what does that mean for tiny or densely packed objects (several objects competing for the same location)?
- Which of the tuned settings helped, by how much, and for which object size? Is the gain larger than the noise of a short single-seed run?
- Compare with Faster R-CNN and RetinaNet by size (AP_small / AP_medium / AP_large) and by speed.

## 8. Query-based: RT-DETR

RT-DETR-L (COCO-pretrained, Ultralytics) with the same data, the same wall-clock budget and the same evaluator. A fixed set of decoder queries predicts one box each and is matched to the
ground truth one-to-one (Hungarian matching), so duplicate predictions are penalised during training and no NMS is needed at inference.

**Parameter investigated:** the number of queries (default 300) relative to scene density — the evaluation frames hold up to `EVAL_COUNTS.max()` objects (printed in section 3). `rtdetr_nq600` doubles the queries.

In [ ]:
def rtdetr_with_queries(nq):
    base = Path(__import__("ultralytics").__file__).parent / "cfg" / "models" / "rt-detr" / "rtdetr-l.yaml"
    cfg = set_queries(yaml.safe_load(base.read_text()), nq)
    out = WORK / f"rtdetr-l-nq{nq}.yaml"; out.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return RTDETR(str(out)).load("rtdetr-l.pt")

print("objects per evaluation frame: max", int(EVAL_COUNTS.max()), "| 95th percentile", int(np.percentile(EVAL_COUNTS, 95)))
QRY = {
    "rtdetr_base":  dict(make=lambda: RTDETR("rtdetr-l.pt"), params=f"300 queries, imgsz {DET_IMGSZ}"),
    "rtdetr_nq600": dict(make=lambda: rtdetr_with_queries(600), params=f"600 queries, imgsz {DET_IMGSZ}"),
}
if RUN_QUERY:
    for name in QRY_RUN:
        if name != "rtdetr_base" and not RUN_TUNED:
            continue
        run_ultra("query-based", name, QRY[name]["make"], RTDETR, DET_IMGSZ, BATCH_RTDETR, 0.0001, {}, QRY[name]["params"])
show_registry("query-based");

In [ ]:
# validation mAP over training: CNN detector vs query-based detector with the same wall-clock budget
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for name in ["yolo_base", "rtdetr_base", "rtdetr_nq600"]:
    p = RUNS_DIR / name / "results.csv"
    if not p.exists():
        continue
    r = pd.read_csv(p); r.columns = r.columns.str.strip()
    x = r["time"] / 60 if "time" in r.columns else r["epoch"]
    ax.plot(x, r["metrics/mAP50-95(B)"], marker="o", label=name)
ax.set_xlabel("training time, min" if "time" in r.columns else "epoch"); ax.set_ylabel("val mAP50-95 (Ultralytics, in-training)")
ax.set_title("Same budget, same data"); ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "map_over_training.png", dpi=140); plt.show()

### query-based

- Did RT-DETR converge within the budget? What does the curve look like compared with YOLO (slope, noise, final value)? State plainly if it had not converged.
- How does one-to-one matching remove the need for NMS, and how does it change training (e.g. slower start)?
- Small objects and dense scenes: see the density table in section 9. Did more queries help in crowded frames?

## 9. Summary comparison (Your pipeline)

Everything that finished in this notebook, plus the earlier experiments from the main notebook, grouped by **what was changed**: the data, the inference strategy or the model.

In [ ]:
df = show_registry()
if not df.empty:
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(df) + 1.5))
    d = df.sort_values("AP"); ax.barh(d.index, d["AP"]); ax.set_xlabel("AP (COCO, IoU .50:.95), same evaluation frames")
    for i, (a, s) in enumerate(zip(d["AP"], d["AP_small"])):
        ax.text(a, i, f" AP_small {s:.3f}", va="center", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "summary_ap.png", dpi=140); plt.show()

In [ ]:
# sparse vs dense frames and small objects, for every model that saved predictions
ids = EVAL_ROWS["image_id"].to_numpy(); dense = EVAL_COUNTS > np.median(EVAL_COUNTS)
out = {}
for r in REGISTRY:
    res = load_preds(r["name"])
    out[r["name"]] = {"AP all": coco_eval(res, ids)[0]["AP"], "AP sparse": coco_eval(res, ids[~dense])[0]["AP"],
                      "AP dense": coco_eval(res, ids[dense])[0]["AP"], "AP_small": coco_eval(res, ids)[0]["AP_small"]}
if out:
    display(pd.DataFrame(out).T.round(4))

In [ ]:
# earlier results from the main YOLO notebook (copied from its outputs and from the Kaggle public leaderboard page).
# NOTE: the val column comes from the Ultralytics validator, the tables above from pycocotools - the two are close but not identical, do not mix them in one ranking.
earlier = pd.DataFrame([
    # stage,                    what changed,                                          type,       val mAP50-95 (<=1904 px, 429 frames), public LB
    ("main notebook", "YOLO11s 1280 px, NMS 0.7, conf 0.001, 500 boxes",                 "baseline",   0.2474, 0.2202),
    ("main notebook", "NMS IoU 0.5",                                                      "inference",  None,   0.2190),
    ("main notebook", "NMS IoU 0.9",                                                      "inference",  None,   0.2001),
    ("main notebook", "max boxes 700 / 1000 instead of 500",                              "inference",  None,   0.2202),
    ("main notebook", "submission_yolo11s_t4_conf001.csv (check its exact setting)",      "inference",  None,   0.2155),
    ("main notebook", "imgsz 1536, no TTA, NMS 0.7",                                      "inference",  0.2521, None),
    ("main notebook", "imgsz 1536, no TTA, NMS 0.6",                                      "inference",  0.2540, None),
    ("main notebook", "imgsz 1280 + TTA, NMS 0.7",                                        "inference",  0.2514, None),
    ("main notebook", "imgsz 1536 + TTA, NMS 0.7",                                        "inference",  0.2594, None),
    ("main notebook", "imgsz 1664 + TTA, NMS 0.7",                                        "inference",  0.2565, None),
    ("main notebook", "imgsz 1536 + TTA, NMS 0.6 (per-size imgsz: 960 px for small frames)", "inference", 0.2624, 0.26),
], columns=["stage", "change", "changed", "val mAP50-95", "public LB"])
display(earlier)
earlier.to_csv(OUT_DIR / "earlier_results.csv", index=False)